In [10]:
!pip install pyspark

In [11]:
from pyspark.sql import SparkSession

spark = SparkSession.builder\
        .master("local")\
        .appName("Q3")\
        .getOrCreate()
sc = spark.sparkContext

In [12]:
!pip install wget

In [13]:
import wget, csv

base = "https://raw.githubusercontent.com/nongaussian/class-2024-datamining/main/data/World Hapiness Report/"
for y in range(2015, 2020):
    wget.download(base + f"{y}.csv", f"{y}.csv")
    print(y, sc.textFile(f"{y}.csv").first())  # 컬럼 이름 출력

2015 Country,Happiness Rank,Happiness Score,Economy (GDP per Capita),Family,Health (Life Expectancy),Freedom,Trust (Government Corruption)
2016 Country,Happiness Rank,Happiness Score,Economy (GDP per Capita),Family,Health (Life Expectancy),Freedom,Trust (Government Corruption)
2017 Country,Happiness Rank,Happiness Score,Economy (GDP per Capita),Family,Health (Life Expectancy),Freedom,Generosity,Trust (Government Corruption)
2018 Country,Happiness Rank,Happiness Score,Economy (GDP per Capita),Family,Health (Life Expectancy),Freedom,Trust (Government Corruption)
2019 Country,Happiness Rank,Happiness Score,Economy (GDP per Capita),Family,Health (Life Expectancy),Freedom,Trust (Government Corruption)


In [14]:
def load(year):
    return (sc.textFile(f"{year}.csv")
        .filter(lambda line: not line.startswith("Country"))  # 헤더 제거
        .map(lambda line: next(csv.reader([line])))           # 쉼표 분리
        .map(lambda t: (t[0], (year, float(t[2])))))          # (국가, (연도, 행복지수))

rdd = load(2015).union(load(2016)).union(load(2017)).union(load(2018)).union(load(2019))

In [15]:
result = (rdd.groupByKey()                                            # 국가별로 (연도, 점수) 묶기
             .mapValues(dict)                                         # {연도: 점수} 딕셔너리로 변환
             .filter(lambda x: len(x[1]) == 5)                        # 5년 모두 조사한 나라
             .mapValues(lambda d: [d[y] for y in range(2015, 2020)])  # 연도순 점수 리스트
             .filter(lambda x: all(x[1][i] <= x[1][i+1] for i in range(4)))  # 매년 전년도 이상인 나라
             .sortByKey()                                             # 국가 이름순 정렬
             .map(lambda x: x[0])                                     # 국가 이름만 추출
             .collect())                                              # 마지막 action으로 결과 가져오기

print(result)

['Benin', 'Burkina Faso', 'Cambodia', 'Cameroon', 'Chad', 'Congo (Brazzaville)', 'Czech Republic', 'Dominican Republic', 'Estonia', 'Finland', 'Gabon', 'Honduras', 'Hungary', 'Ivory Coast', 'Latvia', 'Malta', 'Mongolia', 'Niger', 'Philippines', 'Poland', 'Portugal', 'Romania', 'Senegal', 'Serbia', 'Slovakia', 'Syria', 'Tajikistan', 'Togo']
